# XGBoost active learning — feature engineering in the notebook

Everything model-specific lives **here in the notebook**:

- **Feature engineering** — text & label embeddings, how a `(text, label)` pair
  becomes a feature vector (`concat` / `diff` / `prod`), and optional PCA.
- **The model** — a plain **XGBoost binary classifier**: given a feature vector
  it predicts whether the text has that label. No epochs, no batches — one
  `XGBClassifier.fit(X, y)` with `n_estimators` trees.

The **only project `.py` file** imported is the active-learning widget
(`active_learner.py`), which just drives the labeling loop, retraining,
evaluation and PSO. Swap in any object exposing `fit(text_embeddings, truth, …)`
and `predict_scores(text_embeddings)` and the widget will use it.

Run `python generate_data.py` once first to create the CSVs.

In [ ]:
import json
import os

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# The ONLY project .py import: the active-learning widget + its data container.
from active_learner import ActiveLearner, EmbeddedDataset

LABEL_SEP = "::"

label_dict = {
    "sentiment": {
        "positive": "expresses approval, happiness, or satisfaction",
        "negative": "expresses disapproval, anger, or frustration",
        "neutral":  "factual or descriptive with no clear emotional valence",
    },
    "topic": {
        "pricing":  "mentions cost, price, fees, value for money",
        "support":  "mentions customer service, help, agents, response time",
        "quality":  "mentions product quality, defects, durability, materials",
        "shipping": "mentions delivery, shipping speed, packaging, arrival",
    },
}
flat_labels = [(c, l, d) for c, labs in label_dict.items() for l, d in labs.items()]
label_keys = [f"{c}{LABEL_SEP}{l}" for c, l, _ in flat_labels]


# ── tiny inline data loading (kept in the notebook) ──────────────────────
def coerce_labels(x):
    if isinstance(x, list):
        return [str(v) for v in x]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if s.startswith("["):
            try:
                return [str(v) for v in json.loads(s)]
            except Exception:
                pass
        return [s]
    return []


def read_csv_labels(path):
    df = pd.read_csv(path)
    if "labels" in df.columns:
        df["labels"] = df["labels"].apply(coerce_labels)
    else:
        df["labels"] = [[] for _ in range(len(df))]
    return df


# Training pool = pre-labeled rows + new unlabeled rows (dedup by text).
if os.path.exists("labeled_train_al.csv"):
    labeled_train = read_csv_labels("labeled_train_al.csv")
    labeled_train = labeled_train[labeled_train["labels"].apply(bool)].reset_index(drop=True)
else:
    labeled_train = pd.DataFrame({"text": [], "labels": []})

unlabeled = read_csv_labels("unlabeled_train.csv")
seen = set(labeled_train["text"].astype(str))
unlabeled = unlabeled[~unlabeled["text"].astype(str).isin(seen)].reset_index(drop=True)

pool_df = pd.concat([labeled_train, unlabeled], ignore_index=True)
test_df = read_csv_labels("labeled_demo.csv")

# ── feature engineering (1/2): embeddings, computed once here ────────────
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


def embed(texts):
    return np.asarray(embedding_model.encode(list(texts), normalize_embeddings=True))


pool_text_embs = embed(pool_df["text"].astype(str))
test_text_embs = embed(test_df["text"].astype(str))
label_embs = embed([f"{l}. {d}" for _, l, d in flat_labels])
print("pool:", pool_text_embs.shape, "| test:", test_text_embs.shape, "| labels:", label_embs.shape)

## Feature engineering (2/2) + the XGBoost binary classifier

`make_features` turns a `(text_emb, label_emb)` pair into one feature vector.
`XGBFeatureClassifier` builds positive/negative pairs from the labels, optionally
applies PCA, and trains a single binary `XGBClassifier`. The widget only ever
calls `fit(...)` and `predict_scores(...)`; everything else here is yours to
edit.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import log_loss
import random
import xgboost as xgb


def make_features(text_emb, label_emb, mode="concat+diff+prod"):
    """Combine a text embedding and a label embedding into one feature vector.

    Broadcasts over leading dims, so `text_emb`/`label_emb` may be (d,) or (n, d).
    """
    parts = []
    if "concat" in mode:
        parts += [text_emb, label_emb]
    if "diff" in mode:
        parts.append(text_emb - label_emb)
    if "prod" in mode:
        parts.append(text_emb * label_emb)
    return np.concatenate(parts, axis=-1)


class XGBFeatureClassifier:
    """Plain XGBoost binary classifier over engineered embedding-pair features.

    The widget calls `fit(text_embeddings, truth)` and
    `predict_scores(text_embeddings)`. Feature engineering (make_features +
    optional PCA) and the negative sampling happen right here in the notebook.
    No epochs / no batches — a single XGBClassifier.fit with `n_estimators` trees.
    """

    def __init__(self, label_keys, label_embeddings, feature_mode="concat+diff+prod",
                 use_pca=False, pca_components=64, n_negatives_per_text=3,
                 n_estimators=200, max_depth=6, learning_rate=0.1,
                 threshold=0.5, random_state=42):
        self.label_keys = list(label_keys)
        self.label_embeddings = np.asarray(label_embeddings)
        self.feature_mode = feature_mode
        self.use_pca = use_pca
        self.pca_components = pca_components
        self.n_negatives_per_text = n_negatives_per_text
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.threshold = threshold
        self.random_state = random_state
        self._xgb = None
        self._pca = None

    # Optional hooks the widget uses if present.
    def set_label_embeddings(self, le):
        self.label_embeddings = np.asarray(le)

    def train_hyperparams(self):
        """Controls the widget renders. `learning_rate` is a typed decimal."""
        return [
            {"name": "n_estimators", "label": "Trees", "kind": "int",
             "default": self.n_estimators, "min": 10, "max": 2000, "step": 10,
             "width": "200px", "description": "Boosting rounds (trees).", "pso": False},
            {"name": "learning_rate", "label": "LR", "kind": "float",
             "default": self.learning_rate, "min": 0.001, "max": 1.0, "step": 0.001,
             "width": "200px", "description": "Type a decimal.", "pso": True},
            {"name": "max_depth", "label": "Depth", "kind": "int",
             "default": self.max_depth, "min": 1, "max": 12, "pso": True},
            {"name": "n_negatives_per_text", "label": "Neg/text", "kind": "int",
             "default": self.n_negatives_per_text, "min": 0, "max": 20, "pso": False},
            {"name": "use_pca", "label": "PCA", "kind": "bool",
             "default": self.use_pca, "pso": True},
            {"name": "pca_components", "label": "PCA k", "kind": "int",
             "default": self.pca_components, "min": 2, "max": 512, "pso": True},
            {"name": "feature_mode", "label": "Features", "kind": "choice",
             "default": self.feature_mode,
             "choices": ["concat", "diff", "prod", "concat+diff", "concat+prod",
                         "diff+prod", "concat+diff+prod"], "pso": True},
        ]

    def _build_xy(self, text_embeddings, truth):
        rng = random.Random(self.random_state)
        X, y = [], []
        L = len(self.label_keys)
        for i in range(len(text_embeddings)):
            pos = [c for c in range(L) if truth[i, c]]
            if not pos:
                continue
            t = text_embeddings[i]
            for c in pos:
                X.append(make_features(t, self.label_embeddings[c], self.feature_mode)); y.append(1)
            negs = [c for c in range(L) if c not in pos]
            rng.shuffle(negs)
            for c in negs[:self.n_negatives_per_text]:
                X.append(make_features(t, self.label_embeddings[c], self.feature_mode)); y.append(0)
        return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.int32)

    def fit(self, text_embeddings, truth, *, eval_text_embeddings=None, eval_truth=None,
            epochs=1, n_estimators=None, learning_rate=None, max_depth=None,
            use_pca=None, pca_components=None, feature_mode=None, n_negatives_per_text=None,
            early_stopping_rounds=None, progress_callback=None, log_to_mlflow=False, **_):
        # Apply hyperparameter overrides coming from the widget controls / PSO.
        for k, v in dict(n_estimators=n_estimators, learning_rate=learning_rate,
                         max_depth=max_depth, use_pca=use_pca, pca_components=pca_components,
                         feature_mode=feature_mode, n_negatives_per_text=n_negatives_per_text).items():
            if v is not None:
                setattr(self, k, v)

        truth = np.asarray(truth, dtype=bool)
        X, y = self._build_xy(text_embeddings, truth)

        def emit(ev, **kw):
            if progress_callback:
                try:
                    progress_callback(ev, **kw)
                except Exception:
                    pass

        if len(X) < 2 or len(np.unique(y)) < 2:
            emit("end", n_batches=0, mean_loss=float("nan"), mean_val_loss=float("nan"))
            return {"mean_loss": float("nan"), "mean_val_loss": float("nan"),
                    "n_pairs": float(len(X)), "trainable_params": 0.0,
                    "total_params": 0.0, "trainable_pct": 0.0}

        Xe = ye = None
        if eval_text_embeddings is not None and eval_truth is not None:
            Xe, ye = self._build_xy(eval_text_embeddings, np.asarray(eval_truth, dtype=bool))
            if len(Xe) == 0:
                Xe = ye = None

        if self.use_pca:
            k = max(2, min(self.pca_components, X.shape[1], X.shape[0]))
            self._pca = PCA(n_components=k, random_state=self.random_state)
            X = self._pca.fit_transform(X).astype(np.float32)
            if Xe is not None:
                Xe = self._pca.transform(Xe).astype(np.float32)
        else:
            self._pca = None

        emit("start", n_pairs=len(X), n_eval_pairs=0 if Xe is None else len(Xe), epochs=1,
             batch_size=self.n_estimators, batches_per_epoch=1,
             trainable_params=int(self.n_estimators), total_params=int(self.n_estimators),
             lr=self.learning_rate)

        cbs = []
        if early_stopping_rounds and Xe is not None:
            cbs.append(xgb.callback.EarlyStopping(rounds=int(early_stopping_rounds),
                                                  save_best=True, maximize=False,
                                                  metric_name="logloss"))
        self._xgb = xgb.XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", tree_method="hist",
            n_estimators=int(self.n_estimators), max_depth=int(self.max_depth),
            learning_rate=float(self.learning_rate), random_state=self.random_state,
            callbacks=cbs)
        eval_set = [(X, y)] + ([(Xe, ye)] if Xe is not None else [])
        self._xgb.fit(X, y, eval_set=eval_set, verbose=False)

        ml = float(log_loss(y, self._xgb.predict_proba(X)[:, 1], labels=[0, 1]))
        mvl = (float(log_loss(ye, self._xgb.predict_proba(Xe)[:, 1], labels=[0, 1]))
               if Xe is not None else float("nan"))
        emit("end", n_batches=int(self.n_estimators), mean_loss=ml, mean_val_loss=mvl)
        return {"mean_loss": ml, "mean_val_loss": mvl, "n_pairs": float(len(X)),
                "trainable_params": float(self.n_estimators),
                "total_params": float(self.n_estimators), "trainable_pct": 100.0}

    def predict_scores(self, text_embeddings):
        """(n_texts, n_labels) match probabilities (cosine fallback pre-fit)."""
        te = np.asarray(text_embeddings)
        n, L = len(te), len(self.label_keys)
        if n == 0:
            return np.zeros((0, L))
        if self._xgb is None:
            return te @ self.label_embeddings.T
        X = make_features(np.repeat(te, L, axis=0), np.tile(self.label_embeddings, (n, 1)),
                          self.feature_mode)
        if self._pca is not None:
            X = self._pca.transform(X).astype(np.float32)
        return self._xgb.predict_proba(X)[:, 1].reshape(n, L)

    # Optional PSO state hooks (so each particle starts clean).
    def snapshot_state(self):
        return {"xgb": self._xgb, "pca": self._pca, "threshold": self.threshold}

    def restore_state(self, s):
        if s:
            self._xgb = s.get("xgb")
            self._pca = s.get("pca")
            self.threshold = float(s.get("threshold", self.threshold))

In [ ]:
# Package the embeddings + frames for the widget.
data = EmbeddedDataset(pool_df, pool_text_embs, label_embs, label_dict)
eval_data = EmbeddedDataset(test_df, test_text_embs, label_embs, label_dict)

# The notebook-defined model (feature engineering lives inside it).
model = XGBFeatureClassifier(
    label_keys=label_keys,
    label_embeddings=label_embs,
    feature_mode="concat+diff+prod",
    use_pca=False, pca_components=64,
    n_negatives_per_text=3,
    n_estimators=200, max_depth=6, learning_rate=0.1,
    threshold=0.5,
)

widget = ActiveLearner(
    data=data,
    eval_data=eval_data,
    model=model,
    embedding_model=embedding_model,
    retrain_every=8,
    query_strategy="margin",
    labeled_save_path="labeled_train_al.csv",
)
widget